In [ ]:
!nvidia-smi

In [ ]:
import os
# NOTE!
# this is for a multi-gpu setup. we basically set which GPU is visible for the program.
# if you only have one GPU (most likely the case ;) ), make sure this is set to "0"!
# or remove this line completely!
# otherwise you may accidentally make your GPU "invisible" to the program!
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
# this is important on our server, as the default /tmp directory is very small.
# so if you compile lots of models, it will eventually fill up.
# if you are working on your own machine, you might want/have to remove this!
if os.path.isdir("/var/tmp"):
    os.environ["TMPDIR"] = "/var/tmp"
    os.environ["TORCHINDUCTOR_CACHE_DIR"] = "/var/tmp"

# Variational Autoencoder

In [ ]:
import numpy as np
import torch
from matplotlib import pyplot as plt

from lgm.data import ImageDataModule
from lgm.models.vae import VAETrainingModule, check_code_usage, per_dim_latent_walk, show_latent_distributions
from lgm.ops import interleave
from lgm.utils import interpolate
from lgm.visualization import ImagePlotter

torch.set_float32_matmul_precision('high')
torch.backends.cudnn.benchmark = True

## Part 1: Preamble

VAEs, in practice, are autoencoders with extra steps.
That's the main reason why we already looked at concepts like reconstruction likelihoods for regular autoencoders (we did this only in the exercise!), even tough those aren't proper generative models:
These concepts are also relevant for VAEs, and now we already know about them, and can focus on the additional steps.

### Stochastic Encoder
The main difference in the model is that the encoder is stochastic.
Previously, the encoder just output the latent variables directly.
Now, it outputs a _distribution_ -- just like we learned about for the decoder!
You can technically choose any distribution you want, but by far the most common choice is the Normal/Gaussian.
This is also the only special case we have implemented (`VAEGauss`).
This means that our encoder outputs means and standard deviations for every latent dimension.
Accordingly, it has to output twice as many values as we have latent dimensions!
However, this is already handled in our repository.

There is then an extra step that *samples* latent variables from the distribution using the reparameterization trick.
For a Gaussian posterior, this can be achieved by sampling from a Standard Normal distribution, multiplying by the standard deviation, and adding the mean (in that order!).

### Loss
The distribution parameters returned by the encoder also go into the KL-divergence loss, which encourages encoder outputs to be Standard Normal.
One reason to choose a Gaussian variational posterior, and a Gaussian prior as well, is that the KL-divergence between the two can be computed easily in closed form.
This, in fact, is the main selling point of using the `VAEGauss` special case.
To be precise, the formula is

$$0.5 \sum_i\left(1 + \log(\sigma_i^2) - \mu_i^2  - \sigma_i^2\right)$$

where $i$ indexes the latent dimensions, and $\mu_i, \sigma_i$ are returned by the encoder.
This means the KL-divergence can be computed easily based on the encoder output parameters (*not* samples!).

As for the reconstruction loss, we can re-use the `likelihoods` module from our autoencoder.
In this case, it's very important that all the scaling factors etc. are correct, since otherwise, the reconstruction loss will not be scaled correctly compared to the KL-divergence.
This means you should _not_ use `MeanSquaredError`, as this corresponds to a "naive" reconstruction loss without a proper likelihood function!
In case you are still confused about the whole "reconstruction loss from likelihood" thing (understandable), keep in mind we also have a blog post on this: [LINK](https://ovgu-ailab.github.io/blog/methods/2025/10/05/vae-reconstruction.html)

### VAE Variants
Note that we also implement a generic `VAE` base class that can take any posterior & prior distribution.
The KL-divergence can then simply be approximated by Monte Carlo sampling, just as we are doing for the reconstruction loss.
You would need to define new classes that implement the interface of `PosteriorFamily`, though.
You could, however, try it with the `GaussianPosterior` -- this would implement the standard Gaussian VAE, but estimating the KL divergence from samples rather than the analytic solution.

### Important Parameters
The main parameters to tune, besides the ones we have for the regular autoencoder, are the latent space size, and $\beta$, the weight on the KL-loss.
- Because the KL-loss punishes the model for putting information into the latent space, it's actually next to impossible to make the latent space "too large".
  If you increase it beyond what is "permitted" by the loss, additional dimensions will usually go unused.
  That doesn't mean you should increase it to some absurd value, though;
  more dimensions means more parameters, more memory used, more computation time, etc.
  It just means that the exact choice is not so critical, and overfitting due to the latent space being too large is very difficult.
- $\beta$ is extremely important to tune.
    - Too small, and the model is not properly encouraged to encode data into a Standard Normal (i.e. close to the prior).
      This will usually lead to good reconstructions, but terrible generations.
    - Too large, and you will get "posterior collapse", where the model cannot put enough information into the latent space, and reconstructions and generations will both be a blurry mess.

We recommend that you run multiple experiments, trying a wide range of values for $\beta$ spanning multiple orders of magnitude.
It also heavily interacts with the chosen reconstruction likelihood, so you might want to stick with one for now, and only vary $\beta$.
An alternative approach could be to first try a bunch of likelihoods using $\beta=1$, see which one works best, then stick with that likelihood while varying $\beta$.
This should give you the best starting point for finding a good value relatively quickly.

Note that we also provide the option to take multiple samples from each posterior $q(z|x)$.
The standard is to take just one.
This is implemented via the `samples_per_q` argument in the model/training code.
In case this is confusing to you, you can pretty much ignore this argument, and just leave it at the default value of 1.
In fact, increasing this value seems to not be worth it in most cases.

### Config
Since VAEs are proper generative models, *and* also act as autoencoders, we include callbacks for logging both reconstructions and generations during training.
As in previous models, you can check training progress in Tensorboard!
Furthermore, just as for autoencoders, we dequantize the data.
This is a crucial step for continuous likelihood-based models, and may or may not have been discused in the exercise, depending on the available time. :)


### Addendums to the Exercise Session
Here are a few clarifications we did not get to in the exercise sessions.
We will discuss these in class in the coming week.

#### Summing Loss Over Data Dimensions
If you inspect the code for the reconstruction likelihoods and the associated loss functions, you will notice that we are *summing* over data dimensions, and only average over the batch axis.
This is different from typical losses you may be familiar with, such as mean squared error, where we _average_ over data dimensions.
This difference comes from the correspondence of loss functions to data likelihoods, which are *sums of log probabilities*.
Averaging the reconstruction loss would lead to a completely wrong balance between it and the KL-divergence.

#### Bits/dim Metrics
The above point is a bit annoying, as summing over dimensions gives very large loss values that are hard to interpret.
Our code also logs a `"negative_elbo"` metric, which is reported in "bits per dimension", a commonly used metric.
This reports the average loss per data dimension, but the average crucially also includes the KL-divergence.
Check out the logs in Tensorboard: the `"negative_elbo"` and `"vae_loss"` metrics should have pretty much the same shape, even though the actual numbers will be quite different -- this is all just due to different normalization!

#### Parameterizing Encoder Distributions
Say you are using a Gaussian as $q(z|x)$.
This distribution requires means and variances.
However, we are technically free to choose whether our model returns variances, standard deviations,
precisions, log-variances, etc.
For example, if your model returns standard deviations, you can simply take the square to get the variance.

There is no "correct" choice here.
We just have to make sure to be consistent here and convert to whatever values we need.
For example, the sampler (see below) requires the standard deviations.
The KL-divergence requires variances and log-variances.


## Part 2: Reloading

In [ ]:
dataset = "fashion"
name = "baseline"
version = 0

check_path = f"lightning_logs/vae/{dataset}/{name}/version_{version}/checkpoints/last.ckpt"

data_module = ImageDataModule.load_from_checkpoint(check_path)
data_module.prepare_data()
data_module.setup()
plotter = ImagePlotter(transform=data_module.normalizer.invert)

In [ ]:
lightning_module = VAETrainingModule.load_from_checkpoint(check_path)
model = lightning_module.model.eval()
model.compile()
device = lightning_module.device
print("Device is", device)

In [ ]:
print(model)

## Part 3: Exploration

### Generating Images

Generating images is fast and easy!

In [ ]:
n_rows = 10
with torch.inference_mode():
    generations = model.generate(n_rows**2)
_ = plotter.image_grid(generations,
                       title="Generation Examples",
                       n_rows=n_rows)

### Reconstructions
Of course we can also compute some example reconstructions and plot them.
This allows us to see the difference between reconstruction & generation quality.
The code is pretty much lifted from our `ReconstructionLogging` callback, which creates these plots in Tensorboard, as well.

In [ ]:
examples, _ = next(iter(data_module.test_dataloader()))
examples = examples.to(lightning_module.device)[:n_rows**2 // 2]

In [ ]:
with torch.inference_mode():
    reconstructions = model(examples, map_to_expected=True)
    alternating = interleave(examples, reconstructions)
    subtitles = ["Original", "Reconstruction"] * (n_rows // 2)
    subtitles = subtitles + [""] * (n_rows**2 - n_rows)
    figure = plotter.image_grid(alternating,
                                title="Reconstruction Examples",
                                subtitles=subtitles,
                                n_rows=n_rows)

### Latent Space Exploration

First off, we can get (sampled) encodings for the entire dataset and check their distributions.
You will likely see some deviations from the ideal Gaussian "bells", where our approximate `q(z)` doesn't match the prior `p(z)`.

In [ ]:
show_latent_distributions(model, data_module.test_dataloader(), n_rows=8)

#### Interpolation

Just like for standard autoencoders, we can encode two images, interpolate between them, and decode along the way!

In [ ]:
# pick two random images and plot
randind1 = np.random.randint(len(data_module.test_data))
randind2 = np.random.randint(len(data_module.test_data))

# first get random index, then 0 to only get the image (not label)
image1 = data_module.test_data[randind1][0]
image2 = data_module.test_data[randind2][0]
images = torch.stack((image1, image2))

with torch.inference_mode():
    codes = model.sample_codes(images.to(device))
code1, code2 = codes

# see the images we are dealing with here
_ = plotter.image_grid(images,
                       title="",
                       n_rows=1,
                       n_cols=2)

In [ ]:
n_rows = 8
interpolated_codes = interpolate(code1, code2, n_interpolations=n_rows**2)

with torch.inference_mode():
    interpolated_images = model.reconstruct_from_code(interpolated_codes)
_ = plotter.image_grid(interpolated_images,
                       title="Latent Interpolations",
                       n_rows=n_rows, suppress_plots=True)

In [ ]:
plotter.make_animation(f"animations/interpolate_vae_{dataset}_{name}",
                        torch.cat((interpolated_images, 
                                   torch.flip(interpolated_images[1:-1], dims=[0]))),
                        invert_colors=True)

### Latent Space Usage

Finally, here's a somewhat hacky way to check whether the model is actually making use of the latent space capacity.
The VAE framework (with the KL-loss) actually discourages this, with a stronger effect for larger $\beta$.
What we do here is:

- Iterate over latent dimensions.
- For each, take the original code for an image and fix all dimensions except the current one.
- The current dimension is moved in some range (here, -2 to 2), sampling multiple points along the way.
- Each resulting code is decoded.
- I compute the difference in pixel space between successive images.
- For some dimensions, there is next to no difference.
  I conclude that these dimensions are effectively unused.
- For dimensions where the average change is larger than some threshold (determined empirically), I plot the "walk" over that dimension.

An issue here is that I'm only doing this for a single image.
Ideally, we would sample images randomly, or do the entire process for many images.
The threshold for what average difference to regard as significant also depends on many factors and needs to be experimented with.
You can start by plotting all dimensions and their averages, check which dimensions seem to lead to no/very little change, note their average difference, and then set the threshold somewhat above that.

In [ ]:
walk_differences = per_dim_latent_walk(model.reconstruct_from_code, code1, range_lim=2, threshold=10.)

We can also sort the differences and plot them. We can see that there is a sharp dropoff in difference at some point.
This may be seen as a kind of cut-off point which allows us to (very approximately) read off the number of dimensions actually used in the latent space.
This will vary heavily with beta (higher beta -> fewer dimensions used).

In [ ]:
plt.plot(np.arange(1, len(walk_differences)+1), sorted(walk_differences, reverse=True))
plt.xlabel("Dimension index (sorted)")
plt.ylabel("Average difference")
plt.title("Average differences between successive images in latent space walk per dimension.")
plt.show()

plt.loglog(np.arange(1, len(walk_differences)+1), sorted(walk_differences, reverse=True))
plt.xlabel("Dimension index (sorted)")
plt.ylabel("Average difference")
plt.title("Log-Log scaled version")
plt.show()

Here is another way for a more principled investigation:
The KL-divergence measures the difference from the encoder distribution from the prior.
A stronger deviation indicates more information about the input being present in this dimension -- a dimension with a low KL-divergence is mostly similar to standard normal, for any input, and so carries little information about the input.

Since for Gaussian prior/posterior, the KL is just a sum over the latent dimensions, we can simply look at the _per-dimension_ KL-divergence, averaged over a large dataset.
This gives us a more "global" view of latent space usage, as opposed to the single example above.
However, the plot looks extremly similar to the single-example one!

In [ ]:
all_klds = check_code_usage(model, data_module.test_dataloader())
per_dim_average = all_klds.mean(dim=0).cpu().numpy()

plt.plot(np.arange(1, len(per_dim_average)+1), sorted(per_dim_average, reverse=True))
plt.xlabel("Dimension index (sorted)")
plt.ylabel("Value")
plt.title("Average per-dimension KL-divergence over test set")
plt.show()

plt.loglog(np.arange(1, len(per_dim_average)+1), sorted(per_dim_average, reverse=True))
plt.xlabel("Dimension index (sorted)")
plt.ylabel("Value")
plt.title("Average per-dimension KL-divergence over test set (log-log scaled)")
plt.show()